# Step 5D.2f — UNSW-F1: підтвердження причини (feature order) на другому наборі ознак

`step5d2e` довів на F0: `ORIGINAL_UNSW_F0` (explicit list) у ПОТОЧНОМУ середовищі точно
відтворює історичний `best_iteration=33`, `test_ap=0.973652`, тоді як `AUDIT_UNSW_F0`
(algorithmic, `classify_column()`) дає 277 -- при ІДЕНТИЧНИХ даних, seed, гіперпараметрах,
відмінність лише в порядку колонок.

Цей notebook перевіряє, чи той самий механізм пояснює розбіжність UNSW-F1
(historical best_iteration=290, test_ap=0.983969 vs audit reproduction 249).
F1 = F0 + 5 CTI-ознак (`cve_ctx_service`, `cve_ctx_pool_size`, `cve_ctx_epss_coverage`,
`cve_ctx_n_generic_os_matches`, `cve_ctx_n_specific_product_matches`), в ОДНАКОВОМУ порядку
й наборі в обох пайплайнах (це не оскаржувалось раніше) -- тому єдина змінна тут знову
ЛИШЕ порядок F0-частини.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5d2_audit" / "unsw_f1_feature_order_confirmation"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 1. ORIGINAL / AUDIT F0, ті самі списки, що й у `step5d2e` (literal copy / classify_column)

In [2]:
import re

ORIGINAL_UNSW_F0 = [
    'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload',
    'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth',
    'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
    'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service',
]
CTI_ADDITIONS_F1 = [
    'cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_epss_coverage',
    'cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches',
]  # DATASET_CONFIG["UNSW"]["f1_additions"] -- однаковий, той самий порядок в обох пайплайнах

IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)
TIME_COLS_UNSW = {"Stime", "Ltime"}
LABEL_COL_UNSW = "attack_cat"

def classify_column_unsw(col, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)"
    if col == LABEL_COL_UNSW:
        return "target"
    if col in TIME_COLS_UNSW:
        return "time"
    if IP_PATTERN.match(col):
        return "network identifier (IP)"
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)"
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata"
    if col.lower() == "label":
        return "target-derived (redundant)"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)"
    return "predictor"

INCLUDE_ROLES = {"predictor", "predictor (categorical)", "predictor (overlap risk)"}

raw_unsw = pd.read_parquet(ENRICHED_DIR / "unsw_nb15_with_cve_context_dedup.parquet")
roles_unsw = {col: classify_column_unsw(col, raw_unsw) for col in raw_unsw.columns}
AUDIT_UNSW_F0 = [c for c, r in roles_unsw.items() if r in INCLUDE_ROLES]

assert len(ORIGINAL_UNSW_F0) == 41 and len(AUDIT_UNSW_F0) == 41

ORIGINAL_UNSW_F1 = ORIGINAL_UNSW_F0 + CTI_ADDITIONS_F1
AUDIT_UNSW_F1 = AUDIT_UNSW_F0 + CTI_ADDITIONS_F1

assert len(ORIGINAL_UNSW_F1) == 46 and len(AUDIT_UNSW_F1) == 46
print(f"✅ ORIGINAL_UNSW_F1: {len(ORIGINAL_UNSW_F1)} ознак (F0 explicit order + CTI)")
print(f"✅ AUDIT_UNSW_F1: {len(AUDIT_UNSW_F1)} ознак (F0 algorithmic order + CTI)")


✅ ORIGINAL_UNSW_F1: 46 ознак (F0 explicit order + CTI)
✅ AUDIT_UNSW_F1: 46 ознак (F0 algorithmic order + CTI)


## 2. Позиційне порівняння F1-списків (не лише set) -- перевіряємо ТАКОЖ CTI-частину

Раніше для F0 було встановлено: набір ідентичний (set), порядок -- ні (0/41 позицій
збігались). Тут перевіряємо ПОВНИЙ F1-список (46 позицій) явно, а не припускаємо, що
CTI-додатки (позиції 41-45) йдуть в однаковому порядку в обох pipeline лише тому, що обидва
пайплайни використовують той самий явний `CTI_ADDITIONS_F1`-список.

In [3]:
position_rows_f1 = []
for i in range(max(len(ORIGINAL_UNSW_F1), len(AUDIT_UNSW_F1))):
    orig_col = ORIGINAL_UNSW_F1[i] if i < len(ORIGINAL_UNSW_F1) else None
    audit_col = AUDIT_UNSW_F1[i] if i < len(AUDIT_UNSW_F1) else None
    match = orig_col == audit_col
    position_rows_f1.append(dict(position=i, original=orig_col, audit=audit_col, match=match))

position_df_f1 = pd.DataFrame(position_rows_f1)
n_mismatches_f1 = int((~position_df_f1["match"]).sum())
n_mismatches_f0_part = int((~position_df_f1.loc[:40, "match"]).sum())
n_mismatches_cti_part = int((~position_df_f1.loc[41:, "match"]).sum())

print(f"F1 позиційне порівняння (46 позицій): {46 - n_mismatches_f1}/46 збігаються, "
      f"{n_mismatches_f1} розходяться.")
print(f"  -- з них у F0-частині (позиції 0-40): {n_mismatches_f0_part} розходжень "
      f"(очікувано ідентично до step5d2e: {41 - n_mismatches_f0_part}/41 збігались)")
print(f"  -- з них у CTI-частині (позиції 41-45): {n_mismatches_cti_part} розходжень "
      f"(очікувано 0, бо CTI_ADDITIONS_F1 -- той самий explicit список в обох пайплайнах)")

if n_mismatches_cti_part > 0:
    print("\n❗ CTI-частина ТАКОЖ розходиться позиційно -- це нова, не врахована раніше "
          "змінна, яку варто дослідити окремо перед висновком по F1.")
    print(position_df_f1.loc[41:])
else:
    print("\n✅ CTI-частина позиційно ідентична в обох пайплайнах -- розбіжність F1 (якщо "
          "вона підтвердиться нижче) походить виключно з тієї самої F0-частини, що й у step5d2e.")


F1 позиційне порівняння (46 позицій): 5/46 збігаються, 41 розходяться.
  -- з них у F0-частині (позиції 0-40): 41 розходжень (очікувано ідентично до step5d2e: 0/41 збігались)
  -- з них у CTI-частині (позиції 41-45): 0 розходжень (очікувано 0, бо CTI_ADDITIONS_F1 -- той самий explicit список в обох пайплайнах)

✅ CTI-частина позиційно ідентична в обох пайплайнах -- розбіжність F1 (якщо вона підтвердиться нижче) походить виключно з тієї самої F0-частини, що й у step5d2e.


## 3. Побудова спільного split і обох зрізів (той самий підхід, що й у `step5d2e`)

In [4]:
df_unsw = raw_unsw.copy()
df_unsw["_is_attack"] = (df_unsw["attack_cat"] != "Normal").astype(int)
df_unsw["ct_ftp_cmd"] = pd.to_numeric(df_unsw["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

BOUNDARY_1 = pd.Timestamp("2015-02-18 05:36:51")
BOUNDARY_2 = pd.Timestamp("2015-02-18 08:58:36")
t_unsw = pd.to_datetime(df_unsw["Stime"], unit="s")

train_mask = t_unsw < BOUNDARY_1
val_mask = (t_unsw >= BOUNDARY_1) & (t_unsw < BOUNDARY_2)
test_mask = t_unsw >= BOUNDARY_2

train_df = df_unsw[train_mask].copy()
val_df = df_unsw[val_mask].copy()
test_df = df_unsw[test_mask].copy()

assert len(test_df) == 308_935 and int(test_df["_is_attack"].sum()) == 25_354

categorical_candidates = ["state", "proto", "service"]
if "cve_ctx_service" in df_unsw.columns:
    categorical_candidates.append("cve_ctx_service")

for col in categorical_candidates:
    cats = pd.Categorical(train_df[col]).categories
    train_df[col] = pd.Categorical(train_df[col], categories=cats)
    val_df[col] = pd.Categorical(val_df[col], categories=cats)
    test_df[col] = pd.Categorical(test_df[col], categories=cats)

print(f"✅ Спільний split побудовано: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")

X_train_orig, X_val_orig, X_test_orig = train_df[ORIGINAL_UNSW_F1], val_df[ORIGINAL_UNSW_F1], test_df[ORIGINAL_UNSW_F1]
X_train_aud, X_val_aud, X_test_aud = train_df[AUDIT_UNSW_F1], val_df[AUDIT_UNSW_F1], test_df[AUDIT_UNSW_F1]
y_train, y_val, y_test = train_df["_is_attack"], val_df["_is_attack"], test_df["_is_attack"]

print("\nORIGINAL F1 column order:")
print(" ", list(X_train_orig.columns))
print("\nAUDIT F1 column order:")
print(" ", list(X_train_aud.columns))


✅ Спільний split побудовано: train=1,441,568  val=308,915  test=308,935

ORIGINAL F1 column order:
  ['state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service', 'cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_epss_coverage', 'cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches']

AUDIT F1 column order:
  ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt

## 4. Тренування F1 з обома порядками колонок -- вирішальний тест

In [5]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
SCALE_POS_WEIGHT = n_neg_train / n_pos_train

LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
    scale_pos_weight=SCALE_POS_WEIGHT,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True


def train_with_order(X_train_ord, X_val_ord, X_test_ord, cat_cols):
    lgb_train = lgb.Dataset(X_train_ord, label=y_train, categorical_feature=cat_cols)
    lgb_val = lgb.Dataset(X_val_ord, label=y_val, reference=lgb_train, categorical_feature=cat_cols)

    model = lgb.train(
        LGB_PARAMS, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    best_iter = model.best_iteration
    test_scores = model.predict(X_test_ord, num_iteration=best_iter)
    test_ap = average_precision_score(y_test, test_scores)
    return dict(best_iteration=best_iter, test_ap=test_ap)


cat_cols_orig = [c for c in ORIGINAL_UNSW_F1 if X_train_orig[c].dtype.name == "category"]
cat_cols_aud = [c for c in AUDIT_UNSW_F1 if X_train_aud[c].dtype.name == "category"]

print("=" * 20, "ORIGINAL порядок колонок (F1)", "=" * 20)
result_orig = train_with_order(X_train_orig, X_val_orig, X_test_orig, cat_cols_orig)
bi_orig, tap_orig = result_orig["best_iteration"], result_orig["test_ap"]
print(f"best_iteration={bi_orig}  test_ap={tap_orig:.6f}")

print("\n" + "=" * 20 + " AUDIT порядок колонок (F1) " + "=" * 20)
result_aud = train_with_order(X_train_aud, X_val_aud, X_test_aud, cat_cols_aud)
bi_aud, tap_aud = result_aud["best_iteration"], result_aud["test_ap"]
print(f"best_iteration={bi_aud}  test_ap={tap_aud:.6f}")


==================== ORIGINAL порядок колонок (F1) ====================
[LightGBM] [Info] Number of positive: 49346, number of negative: 1392222
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.176577 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5779
[LightGBM] [Info] Number of data points in the train set: 1441568, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034231 -> initscore=-3.339800
[LightGBM] [Info] Start training from score -3.339800
best_iteration=290  test_ap=0.983969

==================== AUDIT порядок колонок (F1) ====================
[LightGBM] [Info] Number of positive: 49346, number of negative: 1392222
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070470 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

## 5. Порівняння з історичними значеннями UNSW-F1

In [6]:
HISTORICAL_F1_BEST_ITERATION = 290
HISTORICAL_F1_TEST_AP = 0.983969

print(f"result_orig (ORIGINAL порядок, ПОТОЧНЕ середовище): best_iteration={bi_orig}  test_ap={tap_orig:.6f}")
print(f"Історичний Step 5D.2 UNSW-F1: best_iteration={HISTORICAL_F1_BEST_ITERATION}  test_ap={HISTORICAL_F1_TEST_AP:.6f}")
print(f"Попередній audit-reproduction (Step 5D.2b), для довідки: best_iteration=249")

if bi_orig == HISTORICAL_F1_BEST_ITERATION:
    print(
        "\n🎯 ORIGINAL порядок колонок відтворює історичний best_iteration=290 і для F1 -- "
        "підтверджує на другому наборі ознак, що причина розбіжності Step 5D.2b -- саме "
        "порядок колонок F0-частини, а не execution environment."
    )
else:
    print(
        "\n❗ ORIGINAL порядок колонок НЕ відтворює 290 для F1, хоча відтворив 33 для F0 "
        "(step5d2e) -- це означало б, що причина F1-розбіжності частково інша, або що є "
        "додаткова чутливість специфічна для F1/CTI-частини. Потребує окремого дослідження."
    )


result_orig (ORIGINAL порядок, ПОТОЧНЕ середовище): best_iteration=290  test_ap=0.983969
Історичний Step 5D.2 UNSW-F1: best_iteration=290  test_ap=0.983969
Попередній audit-reproduction (Step 5D.2b), для довідки: best_iteration=249

🎯 ORIGINAL порядок колонок відтворює історичний best_iteration=290 і для F1 -- підтверджує на другому наборі ознак, що причина розбіжності Step 5D.2b -- саме порядок колонок F0-частини, а не execution environment.


## Збереження артефактів

In [7]:
summary = dict(
    result_original_order=dict(best_iteration=int(bi_orig), test_ap=float(tap_orig)),
    result_audit_order=dict(best_iteration=int(bi_aud), test_ap=float(tap_aud)),
    historical_best_iteration=HISTORICAL_F1_BEST_ITERATION,
    historical_test_ap=HISTORICAL_F1_TEST_AP,
    original_order_matches_historical=bool(bi_orig == HISTORICAL_F1_BEST_ITERATION),
)
with open(AUDIT_OUT_DIR / "unsw_f1_feature_order_confirmation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit/unsw_f1_feature_order_confirmation:
  unsw_f1_feature_order_confirmation_summary.json


## Підсумок

Якщо цей notebook підтверджує ORIGINAL-порядок → 290/0.983969 для F1, разом з `step5d2e`
(33/0.973652 для F0) це дає повну, симетричну, доказову відповідь на питання, чому
Step 5D.2b не відтворив UNSW: **не execution environment, не LightGBM execution-path
нестабільність (Total Bins/deterministic), а порядок колонок у вхідній матриці ознак**,
який виник через структурну відмінність між explicit-list (`DATASET_CONFIG`) і algorithmic
(`classify_column()`) способами побудови F0.

Практичний висновок для frozen-протоколу: майбутні незалежні реконструкції мають
використовувати ЯВНИЙ, зафіксований порядок ознак (як в оригіналі), а не покладатися на
природний порядок колонок вихідного parquet-файлу, оскільки LightGBM у цьому середовищі
демонстративно чутливий до порядку колонок навіть при ідентичному наборі ознак, даних,
seed і гіперпараметрах.